In [0]:
%sql
CREATE TABLE IF NOT EXISTS oulad.validation.data_quality_results (
    test_timestamp TIMESTAMP,
    table_name STRING,
    test_type STRING,
    column_name STRING,
    grain_columns STRING,
    total_rows BIGINT,
    failed_rows BIGINT,
    test_result STRING
);

In [0]:
%sql
INSERT INTO oulad.validation.data_quality_results
SELECT
    CURRENT_TIMESTAMP() AS test_timestamp,
    'fact_assessments' AS table_name,
    'NOT_NULL' AS test_type,
    'id_student' AS column_name,
    NULL AS grain_columns,
    COUNT(*) AS total_rows,
    COUNT(*) - COUNT(id_student) AS failed_rows,
    CASE
        WHEN COUNT(*) - COUNT(id_student) = 0 THEN 'PASS'
        ELSE 'FAIL'
    END AS test_result
FROM oulad.mart.fact_assessments;

In [0]:
%sql
INSERT INTO oulad.validation.data_quality_results
SELECT
    CURRENT_TIMESTAMP() AS test_timestamp,
    'fact_vle_interactions' AS table_name,
    'NOT_NULL' AS test_type,
    'id_student' AS column_name,
    NULL AS grain_columns,
    COUNT(*) AS total_rows,
    COUNT(*) - COUNT(id_student) AS failed_rows,
    CASE
        WHEN COUNT(*) - COUNT(id_student) = 0 THEN 'PASS'
        ELSE 'FAIL'
    END AS test_result
FROM oulad.mart.fact_vle_interactions;

In [0]:
%sql
INSERT INTO oulad.validation.data_quality_results
SELECT
    CURRENT_TIMESTAMP() AS test_timestamp,
    'fact_assessments' AS table_name,
    'UNIQUENESS' AS test_type,
    NULL AS column_name,
    'id_student,id_assessment' AS grain_columns,
    (SELECT COUNT(*) FROM oulad.mart.fact_assessments) AS total_rows,
    COUNT(*) AS failed_rows,
    CASE
        WHEN COUNT(*) = 0 THEN 'PASS'
        ELSE 'FAIL'
    END AS test_result
FROM (
    SELECT
        id_student,
        id_assessment
    FROM oulad.mart.fact_assessments
    GROUP BY
        id_student,
        id_assessment
    HAVING COUNT(*) > 1
) d;

In [0]:
%sql
INSERT INTO oulad.validation.data_quality_results
SELECT
    CURRENT_TIMESTAMP() AS test_timestamp,
    'fact_vle_interactions' AS table_name,
    'UNIQUENESS' AS test_type,
    NULL AS column_name,
    'id_student,id_site,date' AS grain_columns,
    (SELECT COUNT(*) FROM oulad.mart.fact_vle_interactions) AS total_rows,
    COUNT(*) AS failed_rows,
    CASE
        WHEN COUNT(*) = 0 THEN 'PASS'
        ELSE 'FAIL'
    END AS test_result
FROM (
    SELECT
        id_student,
        id_site,
        date
    FROM oulad.mart.fact_vle_interactions
    GROUP BY
        id_student,
        id_site,
        date
    HAVING COUNT(*) > 1
) d;

In [0]:
%sql
SELECT *
FROM oulad.validation.data_quality_results
ORDER BY test_timestamp DESC;=